# Day 8 — GPU implementation, numerical equivalence and break-even audit

## tl;dr

**Executed result: PASS (9/9 gates).**

- Device/precision: NVIDIA GeForce RTX 4060 Laptop GPU, float64.
- Shared-input numerical agreement: M0/M1 maximum component difference
  **1.421e-14**; M2/M3 maximum component difference
  **1.110e-15**; batched-Greek maximum difference
  **4.452e-10**.
- Measured steady-state break-even: **N = 8,192**.
  At **N = 131,072**, the measured steady-state
  speed-up is **9.28×**, including transfer
  and payoff reduction.
- M2/M3 remain explicitly hybrid: CUDA handles endpoints, marginal exits and nested
  `q3`; exact non-integer Bessel `h_ij` remains on the CPU.
- Regression status: **19 tests passed**. The primary report figure is
  `outputs/day8_gpu_validation/cpu_gpu_core_evidence.png`.

This notebook is the reader-facing Day 8 audit for the three-asset RC-A
autocallable. It is not a new barrier-probability model: it ports the
validated Day 5–7 interfaces to CUDA, makes the remaining hybrid boundary
explicit, and tests CPU/GPU agreement before reporting speed.

## Context & Methods

The implementation follows the usable engineering conclusions in the Day 8
literature review:

- Bilokon et al. motivate double precision, scrambled Sobol inputs, shared
  path work and batched bump scenarios for Greeks.
- Tian et al. and Murakowski et al. motivate explicit pipeline timings and a
  measured break-even path count rather than a headline speed-up target.
- Kim et al. motivate event-driven observation logic and early termination
  after autocall.
- Collange et al. show why parallel reductions need not be bitwise identical;
  therefore correctness is tested pathwise through shared inputs and then by
  float64 component tolerances.

The direct M0/M1 pipeline is CUDA end-to-end after host random-input
generation. M2/M3 are honestly labelled **hybrid**: endpoint GBM, exact
marginal bridge exits and nested trivariate bridge counts run on CUDA, while
the validated non-integer-order Bessel `h_ij` calculation stays in SciPy on
the CPU because CuPy 14.1 has no `ive` implementation.

### Key Assumptions

- The frozen RC-A contract, component schema, event clocks and bump policy
  are unchanged.
- Every CPU/GPU correctness pair consumes identical float64 independent
  normal innovations and the same bridge bank.
- Random-input generation is timed separately. Transfer time is included in
  GPU wall time; steady-state speed-up excludes neither transfer nor payoff
  reduction.
- Performance results apply to this RTX 4060 Laptop GPU, driver, Python and
  CuPy build. Literature speed-ups are context, not acceptance targets.
- AC-Smooth remains the Day 7 local next-observation diagnostic and is not
  silently promoted into the full-product GPU engine.

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import math
import os
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config" / "core_project_config.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config" / "core_project_config.json").exists():
    raise FileNotFoundError("Run this notebook inside the Applied Project repository")

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from autocallable_direct import make_psd_correlation, monitoring_grid, parse_research_contract
from autocallable_gpu import (
    central_bump_scenarios,
    compare_result_components,
    conditioned_replication_shared_input,
    configure_cupy_cache,
    direct_replication_shared_input,
    generate_endpoint_normal_input,
    generate_normal_input,
    gpu_environment,
    greeks_from_scenario_prices,
    price_scenarios_from_normals,
    warmup_gpu,
)

CONFIG_PATH = PROJECT_ROOT / "config" / "core_project_config.json"
VALIDATION_PATH = PROJECT_ROOT / "config" / "day8_gpu_validation.json"
with CONFIG_PATH.open(encoding="utf-8") as handle:
    CONFIG = json.load(handle)
with VALIDATION_PATH.open(encoding="utf-8") as handle:
    DAY8 = json.load(handle)

OUTPUT_DIR = PROJECT_ROOT / DAY8["evidence_directory"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
configure_cupy_cache(OUTPUT_DIR / ".cupy_cache")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.10g}".format
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "font.size": 9.5,
    "axes.titlesize": 11.5,
    "axes.labelsize": 9.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

print("Project root:", PROJECT_ROOT)
print("Evidence directory:", OUTPUT_DIR)

## Data

The market-input workbook and frozen RC-A configuration are reused directly.
The workbook hash is checked before any valuation. Spot ratios are current
levels divided by issuer initial references; the rate, dividends,
volatilities and historical correlation follow the Day 5–7 loader.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


def is_number(value):
    return (
        isinstance(value, (int, float, np.integer, np.floating))
        and not isinstance(value, bool)
        and np.isfinite(value)
    )


def read_market_inputs(path):
    workbook = openpyxl.load_workbook(path, data_only=True, read_only=False)
    setup = workbook["Setup"]
    snapshot = workbook["Underlying_Snapshot"]
    history = workbook["Underlying_History"]
    market_history = workbook["Market_History"]
    issuer_initial = workbook["Issuer_Initial_Value"]
    tickers = [snapshot.cell(row, 2).value for row in (6, 7, 8)]
    spots = np.array([snapshot.cell(row, 4).value for row in (6, 7, 8)], dtype=float)
    dividend_yields = np.array([snapshot.cell(row, 5).value for row in (6, 7, 8)], dtype=float) / 100.0
    volatilities = np.array([snapshot.cell(row, 8).value for row in (6, 7, 8)], dtype=float) / 100.0
    initial_references = np.array([issuer_initial.cell(row, 3).value for row in (38, 39, 40)], dtype=float)
    histories = []
    for date_column, value_column, ticker in zip((1, 4, 7), (2, 5, 8), tickers):
        values = {}
        for row in range(6, history.max_row + 1):
            date_value = history.cell(row, date_column).value
            level = history.cell(row, value_column).value
            if hasattr(date_value, "year") and is_number(level) and level > 0:
                values[pd.Timestamp(date_value)] = float(level)
        histories.append(pd.Series(values, name=ticker).sort_index())
    prices = pd.concat(histories, axis=1, join="inner").dropna()
    correlation = make_psd_correlation(np.log(prices / prices.shift(1)).dropna().corr().to_numpy())
    rate_rows = []
    for row in range(6, market_history.max_row + 1):
        date_value = market_history.cell(row, 10).value
        rate_pct = market_history.cell(row, 11).value
        if hasattr(date_value, "year") and is_number(rate_pct) and rate_pct > 0:
            rate_rows.append((pd.Timestamp(date_value), float(rate_pct) / 100.0))
    rates = pd.Series(dict(rate_rows)).sort_index()
    return {
        "tickers": tickers,
        "spots": spots,
        "dividend_yields": dividend_yields,
        "volatilities": volatilities,
        "initial_references": initial_references,
        "initial_spot_ratios": spots / initial_references,
        "correlation": correlation,
        "risk_free_rate": float(rates.iloc[-1]),
        "rate_as_of": rates.index[-1],
        "workbook_as_of": pd.Timestamp(setup["B8"].value),
    }


workbook_path = PROJECT_ROOT / CONFIG["market_data"]["relative_path"]
workbook_hash = sha256_file(workbook_path)
assert workbook_hash == CONFIG["market_data"]["sha256"], "Frozen workbook hash mismatch"
market = read_market_inputs(workbook_path)
contract = parse_research_contract(CONFIG, DAY8["contract_id"])

market_inputs = pd.DataFrame({
    "ticker": market["tickers"],
    "spot": market["spots"],
    "initial_reference": market["initial_references"],
    "initial_spot_ratio": market["initial_spot_ratios"],
    "dividend_yield": market["dividend_yields"],
    "volatility": market["volatilities"],
})
market_inputs.to_csv(OUTPUT_DIR / "market_inputs.csv", index=False)
pd.DataFrame(
    market["correlation"], index=market["tickers"], columns=market["tickers"]
).to_csv(OUTPUT_DIR / "correlation_matrix.csv")
display(market_inputs)
display(pd.DataFrame(market["correlation"], index=market["tickers"], columns=market["tickers"]))
print(f"Workbook hash PASS; as of {market['workbook_as_of'].date()}; rate {market['risk_free_rate']:.4%} as of {market['rate_as_of'].date()}")

In [ ]:
environment_record = gpu_environment()
if not environment_record["cupy_available"]:
    raise RuntimeError(environment_record.get("unavailable_reason", "CUDA unavailable"))
simple_warmup = warmup_gpu()
environment_record.update(simple_warmup)
environment_record.update({
    "workbook_sha256": workbook_hash,
    "workbook_as_of": market["workbook_as_of"].date().isoformat(),
    "validation_config": str(VALIDATION_PATH.relative_to(PROJECT_ROOT)),
})
environment_table = pd.DataFrame(
    [{"field": key, "value": value} for key, value in environment_record.items()]
)
environment_table.to_csv(OUTPUT_DIR / "environment.csv", index=False)
display(environment_table)

## Results

### 1. M0/M1 shared-input numerical equivalence

Every method below generates one host input, then passes it unchanged to the
NumPy and CuPy implementations. Component values, identities and probability
mass are checked separately; a close total price cannot hide offsetting
component errors.

In [ ]:
correctness_cfg = DAY8["correctness"]
tolerance_cfg = DAY8["tolerances"]
correctness_rows = []
direct_run_rows = []
direct_result_registry = []
for method, method_label in (("mc", "M0"), ("rqmc", "M1")):
    grid, _ = monitoring_grid(contract, correctness_cfg["steps_per_year"])
    normal_input = generate_normal_input(
        correctness_cfg["n_paths"],
        len(grid) - 1,
        method,
        correctness_cfg["seed"] + (0 if method == "mc" else 1),
    )
    common = dict(
        contract=contract,
        risk_free_rate=market["risk_free_rate"],
        dividend_yields=market["dividend_yields"],
        volatilities=market["volatilities"],
        correlation=market["correlation"],
        annual_coupon=contract.baseline_annual_coupon,
        n_paths=correctness_cfg["n_paths"],
        steps_per_year=correctness_cfg["steps_per_year"],
        method=method,
        seed=normal_input.seed,
        batch_size=correctness_cfg["batch_size"],
        initial_spot_ratios=market["initial_spot_ratios"],
        normal_input=normal_input,
    )
    cpu_result, _ = direct_replication_shared_input(**common, backend="cpu")
    gpu_result, _ = direct_replication_shared_input(**common, backend="gpu")
    direct_result_registry.extend([cpu_result, gpu_result])
    comparison = compare_result_components(cpu_result, gpu_result)
    comparison["method"] = method_label
    comparison["tolerance"] = tolerance_cfg["direct_component_absolute"]
    comparison["pass"] = comparison["absolute_difference"] <= comparison["tolerance"]
    correctness_rows.append(comparison)
    for result in (cpu_result, gpu_result):
        direct_run_rows.append({
            key: result.get(key)
            for key in (
                "method", "backend", "n_paths", "steps_per_year", "grid_steps",
                "random_input_seconds", "setup_seconds", "transfer_seconds",
                "pricing_kernel_seconds", "reduction_seconds", "total_wall_seconds",
                "paths_per_second", "peak_memory_pool_bytes", "component_identity_error",
                "probability_mass_error", "fair_coupon_residual", "total_value",
            )
        })

direct_correctness = pd.concat(correctness_rows, ignore_index=True)
direct_timings = pd.DataFrame(direct_run_rows)
direct_correctness.to_csv(OUTPUT_DIR / "cpu_gpu_correctness.csv", index=False)
direct_timings.to_csv(OUTPUT_DIR / "direct_run_timings.csv", index=False)
display(direct_correctness.groupby("method").agg(max_abs_difference=("absolute_difference", "max"), all_pass=("pass", "all")))
display(direct_timings)

### 2. Cold-start, steady-state throughput and break-even N

The first GPU price at each shape is retained as a cold/shape-setup result;
the same input is then priced again for steady-state timing. CPU and GPU
totals both include payoff reduction, and the GPU total includes H2D
transfer. Random generation remains a separate reported field.

In [ ]:
performance_cfg = DAY8["performance"]
performance_rows = []
for n_paths in performance_cfg["path_grid"]:
    grid, _ = monitoring_grid(contract, performance_cfg["steps_per_year"])
    normal_input = generate_normal_input(
        n_paths,
        len(grid) - 1,
        "mc",
        performance_cfg["seed_base"] + n_paths,
    )
    common = dict(
        contract=contract,
        risk_free_rate=market["risk_free_rate"],
        dividend_yields=market["dividend_yields"],
        volatilities=market["volatilities"],
        correlation=market["correlation"],
        annual_coupon=contract.baseline_annual_coupon,
        n_paths=n_paths,
        steps_per_year=performance_cfg["steps_per_year"],
        method="mc",
        seed=normal_input.seed,
        batch_size=min(performance_cfg["batch_size"], n_paths),
        initial_spot_ratios=market["initial_spot_ratios"],
        normal_input=normal_input,
    )
    cpu_result, _ = direct_replication_shared_input(**common, backend="cpu")
    gpu_first, _ = direct_replication_shared_input(**common, backend="gpu")
    gpu_steady, _ = direct_replication_shared_input(**common, backend="gpu")
    performance_rows.append({
        "n_paths": n_paths,
        "grid_steps": cpu_result["grid_steps"],
        "normal_input_seconds": normal_input.generation_seconds,
        "normal_input_bytes": normal_input.values.nbytes,
        "cpu_total_seconds": cpu_result["total_wall_seconds"],
        "gpu_first_call_seconds": gpu_first["total_wall_seconds"],
        "gpu_steady_seconds": gpu_steady["total_wall_seconds"],
        "gpu_transfer_seconds": gpu_steady["transfer_seconds"],
        "gpu_pricing_kernel_seconds": gpu_steady["pricing_kernel_seconds"],
        "gpu_reduction_seconds": gpu_steady["reduction_seconds"],
        "cpu_paths_per_second": cpu_result["paths_per_second"],
        "gpu_paths_per_second": gpu_steady["paths_per_second"],
        "steady_speedup": cpu_result["total_wall_seconds"] / gpu_steady["total_wall_seconds"],
        "cold_speedup": cpu_result["total_wall_seconds"] / gpu_first["total_wall_seconds"],
        "gpu_peak_memory_pool_bytes": gpu_steady["peak_memory_pool_bytes"],
        "absolute_price_difference": abs(cpu_result["total_value"] - gpu_steady["total_value"]),
    })
    del normal_input, cpu_result, gpu_first, gpu_steady
    gc.collect()

performance = pd.DataFrame(performance_rows)
break_even_candidates = performance.loc[performance["steady_speedup"] > 1.0, "n_paths"]
break_even_n = int(break_even_candidates.min()) if len(break_even_candidates) else None
performance.to_csv(OUTPUT_DIR / "performance_scaling.csv", index=False)
display(performance)
print("Measured steady-state break-even N:", break_even_n)

In [ ]:
REPORT_COLORS = {
    "navy": "#173F5F",
    "blue": "#20639B",
    "teal": "#2A9D8F",
    "amber": "#E9A23B",
    "red": "#C44536",
    "slate": "#64748B",
    "light_slate": "#CBD5E1",
    "ink": "#1E293B",
}
REPORT_STYLE = {
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.titlesize": 11,
    "axes.titleweight": "semibold",
    "axes.labelsize": 9.5,
    "axes.edgecolor": "#475569",
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#CBD5E1",
    "grid.alpha": 0.45,
    "grid.linewidth": 0.65,
    "legend.frameon": False,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "text.color": "#1E293B",
}

path_values = performance["n_paths"].to_numpy()
path_labels = [f"{int(value):,}" for value in path_values]
with plt.rc_context(REPORT_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.55), constrained_layout=True)

    axes[0].plot(
        path_values, performance["cpu_total_seconds"],
        color=REPORT_COLORS["slate"], marker="o", markersize=5.5,
        linewidth=2.0, label="CPU total"
    )
    axes[0].plot(
        path_values, performance["gpu_first_call_seconds"],
        color=REPORT_COLORS["amber"], marker="s", markersize=5.0,
        linewidth=1.7, linestyle="--", label="GPU first call"
    )
    axes[0].plot(
        path_values, performance["gpu_steady_seconds"],
        color=REPORT_COLORS["blue"], marker="o", markersize=5.5,
        linewidth=2.2, label="GPU steady state"
    )
    axes[0].axvspan(
        break_even_n, path_values.max(), color=REPORT_COLORS["teal"], alpha=0.07
    )
    axes[0].axvline(
        break_even_n, color=REPORT_COLORS["teal"], linestyle=":", linewidth=1.2
    )
    axes[0].set_xscale("log", base=2)
    axes[0].set_yscale("log")
    axes[0].set_xticks(path_values, path_labels, rotation=0)
    axes[0].set_xlabel("Number of paths, N")
    axes[0].set_ylabel("Wall-clock time (seconds, log scale)")
    axes[0].set_title("(a) Cold-start and steady-state runtime", loc="left")
    axes[0].legend(loc="upper left", ncol=1)
    axes[0].text(
        break_even_n * 1.08, axes[0].get_ylim()[0] * 1.35,
        f"Break-even\nN = {break_even_n:,}",
        color=REPORT_COLORS["teal"], fontsize=8.5, va="bottom"
    )

    axes[1].plot(
        path_values, performance["steady_speedup"],
        color=REPORT_COLORS["teal"], marker="o", markersize=5.5,
        linewidth=2.2, label="Steady-state speed-up"
    )
    axes[1].axhline(1.0, color=REPORT_COLORS["red"], linewidth=1.1, linestyle="--")
    axes[1].fill_between(
        path_values, 1.0, performance["steady_speedup"],
        where=performance["steady_speedup"].to_numpy() >= 1.0,
        color=REPORT_COLORS["teal"], alpha=0.10, interpolate=True
    )
    axes[1].set_xscale("log", base=2)
    axes[1].set_xticks(path_values, path_labels, rotation=0)
    axes[1].set_xlabel("Number of paths, N")
    axes[1].set_ylabel("CPU time / GPU steady-state time")
    axes[1].set_title("(b) Measured steady-state acceleration", loc="left")
    axes[1].annotate(
        f"{performance.iloc[-1]['steady_speedup']:.1f}×",
        xy=(path_values[-1], performance.iloc[-1]["steady_speedup"]),
        xytext=(-30, 15), textcoords="offset points",
        color=REPORT_COLORS["teal"], fontweight="semibold",
        arrowprops={"arrowstyle": "->", "color": REPORT_COLORS["teal"], "lw": 1.0}
    )

    fig.suptitle(
        "CPU/GPU Runtime Scaling and Measured Break-even",
        fontsize=14, fontweight="semibold", y=1.03
    )
    fig.text(
        0.5, -0.035,
        "Shared float64 path inputs. GPU totals include host-to-device transfer and payoff reduction; random-input generation is reported separately.",
        ha="center", fontsize=8, color=REPORT_COLORS["slate"]
    )
    fig.savefig(
        OUTPUT_DIR / "cpu_gpu_performance_scaling.png",
        dpi=300, bbox_inches="tight", pad_inches=0.12
    )
    plt.show()

### 3. Batched CRN bump scenarios and Greeks

Thirteen scenarios (base, three spot pairs and three volatility pairs)
share one scrambled-Sobol input. They are evaluated together on a scenario
axis, so random-input transfer and correlated shocks are reused.

In [ ]:
greeks_cfg = DAY8["batched_greeks"]
grid, _ = monitoring_grid(contract, greeks_cfg["steps_per_year"])
greek_input = generate_normal_input(
    greeks_cfg["n_paths"], len(grid) - 1, "rqmc", greeks_cfg["seed"]
)
labels, spot_scenarios, volatility_scenarios = central_bump_scenarios(
    market["initial_spot_ratios"],
    market["volatilities"],
    greeks_cfg["spot_relative_bump"],
    greeks_cfg["volatility_absolute_bump"],
)
scenario_common = dict(
    contract=contract,
    risk_free_rate=market["risk_free_rate"],
    dividend_yields=market["dividend_yields"],
    correlation=market["correlation"],
    annual_coupon=contract.baseline_annual_coupon,
    normals=greek_input.values,
    steps_per_year=greeks_cfg["steps_per_year"],
    scenario_labels=labels,
    initial_spot_scenarios=spot_scenarios,
    volatility_scenarios=volatility_scenarios,
    batch_size=greeks_cfg["batch_size"],
)
cpu_scenarios, cpu_scenario_timing = price_scenarios_from_normals(**scenario_common, backend="cpu")
gpu_scenarios_first, gpu_scenario_timing_first = price_scenarios_from_normals(**scenario_common, backend="gpu")
gpu_scenarios, gpu_scenario_timing = price_scenarios_from_normals(**scenario_common, backend="gpu")
cpu_greeks = greeks_from_scenario_prices(
    cpu_scenarios,
    market["initial_spot_ratios"],
    greeks_cfg["spot_relative_bump"],
    greeks_cfg["volatility_absolute_bump"],
)
gpu_greeks = greeks_from_scenario_prices(
    gpu_scenarios,
    market["initial_spot_ratios"],
    greeks_cfg["spot_relative_bump"],
    greeks_cfg["volatility_absolute_bump"],
)
greek_agreement = cpu_greeks.merge(
    gpu_greeks, on=["greek", "asset"], suffixes=("_cpu", "_gpu")
)
greek_agreement["absolute_difference"] = abs(
    greek_agreement["value_gpu"] - greek_agreement["value_cpu"]
)
greek_agreement["tolerance"] = tolerance_cfg["greek_absolute"]
greek_agreement["pass"] = greek_agreement["absolute_difference"] <= greek_agreement["tolerance"]
batched_greeks_timing = pd.DataFrame([
    cpu_scenario_timing,
    {**gpu_scenario_timing_first, "backend": "gpu-first-call"},
    {**gpu_scenario_timing, "backend": "gpu-steady"},
])
pd.concat([cpu_scenarios, gpu_scenarios], ignore_index=True).to_csv(
    OUTPUT_DIR / "greek_scenario_prices.csv", index=False
)
greek_agreement.to_csv(OUTPUT_DIR / "greeks_cpu_gpu_agreement.csv", index=False)
batched_greeks_timing.to_csv(OUTPUT_DIR / "batched_greeks_timing.csv", index=False)
display(greek_agreement)
display(batched_greeks_timing)

### 4. M2/M3 hybrid Brownian-bridge pipeline

This check uses identical endpoint normals and identical conditional bridge
banks. It validates that GPU endpoint evolution, marginal probabilities and
nested exit counts preserve the Day 6 conditional-weight price. The exact
bivariate Bessel component is timed separately on CPU.

In [ ]:
conditioned_cfg = DAY8["conditioned"]
conditioned_comparisons = []
conditioned_run_rows = []
conditioned_segment_frames = []
conditioned_result_registry = []
for method_index, (outer_method, method_label) in enumerate((("mc", "M2"), ("rqmc", "M3"))):
    endpoint_input = generate_endpoint_normal_input(
        conditioned_cfg["n_paths"],
        len(contract.observation_times),
        outer_method,
        conditioned_cfg["seed_base"] + method_index,
    )
    common = dict(
        contract=contract,
        risk_free_rate=market["risk_free_rate"],
        dividend_yields=market["dividend_yields"],
        volatilities=market["volatilities"],
        correlation=market["correlation"],
        annual_coupon=contract.baseline_annual_coupon,
        endpoint_normals=endpoint_input.values,
        outer_method=outer_method,
        seed=endpoint_input.seed,
        bridge_bank_seed=conditioned_cfg["bridge_bank_seed"],
        inner_paths=conditioned_cfg["inner_paths"],
        bridge_substeps=conditioned_cfg["bridge_substeps"],
        bessel_terms=conditioned_cfg["bessel_terms"],
        initial_spot_ratios=market["initial_spot_ratios"],
        return_diagnostics=True,
    )
    cpu_result, cpu_segments, _ = conditioned_replication_shared_input(
        **common, endpoint_backend="cpu", probability_backend="cpu"
    )
    gpu_result, gpu_segments, _ = conditioned_replication_shared_input(
        **common, endpoint_backend="gpu", probability_backend="gpu-hybrid"
    )
    conditioned_result_registry.extend([cpu_result, gpu_result])
    comparison = compare_result_components(cpu_result, gpu_result)
    comparison["method"] = method_label
    comparison["tolerance"] = tolerance_cfg["conditioned_component_absolute"]
    comparison["pass"] = comparison["absolute_difference"] <= comparison["tolerance"]
    conditioned_comparisons.append(comparison)
    for result in (cpu_result, gpu_result):
        conditioned_run_rows.append({
            key: result.get(key)
            for key in (
                "method", "endpoint_backend", "probability_backend", "n_paths",
                "inner_paths", "bridge_substeps", "bessel_terms", "endpoint_h2d_seconds",
                "endpoint_kernel_seconds", "endpoint_d2h_seconds", "probability_g_seconds",
                "probability_h_bessel_cpu_seconds", "probability_q3_seconds",
                "nested_gpu_kernel_seconds", "nested_gpu_transfer_seconds",
                "total_wall_seconds", "total_value", "component_identity_error",
                "probability_mass_error", "fair_coupon_residual", "pipeline_claim_boundary",
            )
        })
    for frame, backend in ((cpu_segments, "cpu"), (gpu_segments, "gpu-hybrid")):
        frame = frame.copy()
        frame["method"] = method_label
        frame["backend"] = backend
        conditioned_segment_frames.append(frame)

conditioned_agreement = pd.concat(conditioned_comparisons, ignore_index=True)
conditioned_timings = pd.DataFrame(conditioned_run_rows)
conditioned_segments = pd.concat(conditioned_segment_frames, ignore_index=True)
conditioned_agreement.to_csv(OUTPUT_DIR / "conditioned_cpu_gpu_agreement.csv", index=False)
conditioned_timings.to_csv(OUTPUT_DIR / "conditioned_run_timings.csv", index=False)
conditioned_segments.to_csv(OUTPUT_DIR / "conditioned_segment_timings.csv", index=False)
display(conditioned_agreement.groupby("method").agg(max_abs_difference=("absolute_difference", "max"), all_pass=("pass", "all")))
display(conditioned_timings)

In [ ]:
gpu_conditioned = conditioned_timings[
    conditioned_timings["probability_backend"] == "gpu-hybrid"
].copy()
timing_columns = [
    "endpoint_kernel_seconds",
    "probability_g_seconds",
    "probability_h_bessel_cpu_seconds",
    "nested_gpu_kernel_seconds",
    "nested_gpu_transfer_seconds",
]
friendly_labels = [
    "Endpoint GBM (GPU)",
    "Marginal exits (GPU)",
    "Exact pair probability (CPU)",
    "Nested q3 kernel (GPU)",
    "Nested transfers",
]
component_colors = [
    REPORT_COLORS["navy"],
    REPORT_COLORS["blue"],
    REPORT_COLORS["amber"],
    REPORT_COLORS["teal"],
    REPORT_COLORS["light_slate"],
]
plot_frame = gpu_conditioned.set_index("method")[timing_columns]
plot_frame.columns = friendly_labels

with plt.rc_context(REPORT_STYLE):
    fig, ax = plt.subplots(figsize=(8.6, 4.65), constrained_layout=True)
    bottom = np.zeros(len(plot_frame))
    positions = np.arange(len(plot_frame))
    for label, color in zip(friendly_labels, component_colors):
        values = plot_frame[label].to_numpy()
        ax.bar(
            positions, values, bottom=bottom, width=0.58,
            color=color, edgecolor="white", linewidth=0.7, label=label
        )
        bottom += values
    for position, total in zip(positions, bottom):
        ax.text(
            position, total + max(bottom) * 0.025, f"{total:.3f}s",
            ha="center", va="bottom", fontsize=8.5, fontweight="semibold"
        )
    ax.set_xticks(positions, plot_frame.index)
    ax.set_ylabel("Measured runtime (seconds)")
    ax.set_title("Conditioned M2/M3 Runtime Decomposition", pad=12)
    ax.legend(
        loc="upper center", bbox_to_anchor=(0.5, -0.13),
        ncol=3, fontsize=8, columnspacing=1.4, handlelength=1.5
    )
    ax.grid(axis="x", visible=False)
    fig.text(
        0.5, -0.03,
        "Hybrid boundary: exact non-integer Bessel pair probabilities remain on CPU; endpoint, marginal and nested-q3 work runs on GPU.",
        ha="center", fontsize=8, color=REPORT_COLORS["slate"]
    )
    fig.savefig(
        OUTPUT_DIR / "conditioned_pipeline_timing.png",
        dpi=300, bbox_inches="tight", pad_inches=0.12
    )
    plt.show()

### 5. Automated gates, tests and audit inventory

Speed-up itself is deliberately not a pass/fail gate. The gates require a
working CUDA device, CPU/GPU numerical agreement, unchanged component and
probability identities, all required timing fields, and a clean regression
test run.

In [ ]:
test_environment = os.environ.copy()
test_environment["CUPY_CACHE_DIR"] = str(OUTPUT_DIR / ".cupy_cache")
test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no"],
    cwd=PROJECT_ROOT,
    env=test_environment,
    text=True,
    capture_output=True,
    timeout=240,
)
pytest_text = test_run.stdout + ("\n" + test_run.stderr if test_run.stderr else "")
(OUTPUT_DIR / "pytest_output.txt").write_text(pytest_text, encoding="utf-8")
print(pytest_text)

all_direct_and_conditioned = direct_result_registry + conditioned_result_registry
identity_max = max(
    max(abs(float(result["component_identity_error"])) for result in all_direct_and_conditioned),
    float(pd.concat([cpu_scenarios, gpu_scenarios])["component_identity_error"].abs().max()),
)
probability_mass_max = max(
    max(abs(float(result["probability_mass_error"])) for result in all_direct_and_conditioned),
    float(pd.concat([cpu_scenarios, gpu_scenarios])["probability_mass_error"].abs().max()),
)
fair_coupon_root_max = max(
    abs(float(result["fair_coupon_residual"])) for result in all_direct_and_conditioned
)
required_direct_fields = {
    "transfer_seconds", "pricing_kernel_seconds", "reduction_seconds",
    "total_wall_seconds", "paths_per_second", "peak_memory_pool_bytes",
}
required_conditioned_fields = {
    "probability_g_seconds", "probability_h_bessel_cpu_seconds",
    "probability_q3_seconds", "nested_gpu_kernel_seconds",
}
timing_fields_present = all(
    required_direct_fields.issubset(result) for result in direct_result_registry if result["backend"] == "gpu"
) and all(
    required_conditioned_fields.issubset(result)
    for result in conditioned_result_registry
    if result["probability_backend"] == "gpu-hybrid"
)

gates = pd.DataFrame([
    {"gate": "CUDA device available", "observed": environment_record["cupy_available"], "limit": True, "pass": bool(environment_record["cupy_available"])},
    {"gate": "M0/M1 maximum component difference", "observed": direct_correctness["absolute_difference"].max(), "limit": tolerance_cfg["direct_component_absolute"], "pass": bool(direct_correctness["pass"].all())},
    {"gate": "Batched Greek maximum difference", "observed": greek_agreement["absolute_difference"].max(), "limit": tolerance_cfg["greek_absolute"], "pass": bool(greek_agreement["pass"].all())},
    {"gate": "M2/M3 maximum component difference", "observed": conditioned_agreement["absolute_difference"].max(), "limit": tolerance_cfg["conditioned_component_absolute"], "pass": bool(conditioned_agreement["pass"].all())},
    {"gate": "Component identity maximum absolute error", "observed": identity_max, "limit": tolerance_cfg["component_identity_absolute"], "pass": identity_max <= tolerance_cfg["component_identity_absolute"]},
    {"gate": "Probability mass maximum absolute error", "observed": probability_mass_max, "limit": tolerance_cfg["probability_mass_absolute"], "pass": probability_mass_max <= tolerance_cfg["probability_mass_absolute"]},
    {"gate": "Fair-coupon root maximum absolute residual", "observed": fair_coupon_root_max, "limit": tolerance_cfg["fair_coupon_root_absolute"], "pass": fair_coupon_root_max <= tolerance_cfg["fair_coupon_root_absolute"]},
    {"gate": "Required timing fields present", "observed": timing_fields_present, "limit": True, "pass": bool(timing_fields_present)},
    {"gate": "Regression tests", "observed": test_run.returncode, "limit": 0, "pass": test_run.returncode == 0},
])
overall_status = "PASS" if gates["pass"].all() else "FAIL"
gates.to_csv(OUTPUT_DIR / "gate_summary.csv", index=False)
display(gates)
print("Day 8 overall status:", overall_status)

In [ ]:
# Presentation-ready core evidence: correctness first, acceleration second.
method_error_rows = []
for method_label, group in direct_correctness.groupby("method"):
    method_error_rows.append({
        "method": method_label,
        "max_absolute_difference": group["absolute_difference"].max(),
        "tolerance": tolerance_cfg["direct_component_absolute"],
    })
method_error_rows.append({
    "method": "Batched Greeks",
    "max_absolute_difference": greek_agreement["absolute_difference"].max(),
    "tolerance": tolerance_cfg["greek_absolute"],
})
for method_label, group in conditioned_agreement.groupby("method"):
    method_error_rows.append({
        "method": method_label,
        "max_absolute_difference": group["absolute_difference"].max(),
        "tolerance": tolerance_cfg["conditioned_component_absolute"],
    })
method_errors = pd.DataFrame(method_error_rows)
method_errors.to_csv(OUTPUT_DIR / "presentation_error_summary.csv", index=False)

# Core two-panel figure.
path_values = performance["n_paths"].to_numpy()
path_labels = [f"{int(value):,}" for value in path_values]
with plt.rc_context(REPORT_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), constrained_layout=True)

    axes[0].plot(
        path_values, performance["cpu_total_seconds"],
        marker="o", markersize=5.5, linewidth=2.1,
        label="CPU total", color=REPORT_COLORS["slate"]
    )
    axes[0].plot(
        path_values, performance["gpu_steady_seconds"],
        marker="o", markersize=5.5, linewidth=2.3,
        label="GPU steady state", color=REPORT_COLORS["blue"]
    )
    axes[0].axvspan(
        break_even_n, path_values.max(), color=REPORT_COLORS["teal"], alpha=0.07
    )
    axes[0].axvline(
        break_even_n, color=REPORT_COLORS["teal"], linestyle=":", linewidth=1.2
    )
    axes[0].set_xscale("log", base=2)
    axes[0].set_yscale("log")
    axes[0].set_xticks(path_values, path_labels)
    axes[0].set_xlabel("Number of paths, N")
    axes[0].set_ylabel("Wall-clock time (seconds, log scale)")
    axes[0].set_title("(a) Runtime scaling", loc="left")
    axes[0].legend(loc="upper left")
    axes[0].text(
        break_even_n * 1.08, axes[0].get_ylim()[0] * 1.35,
        f"Measured break-even\nN = {break_even_n:,}",
        color=REPORT_COLORS["teal"], fontsize=8.5, va="bottom"
    )
    axes[0].annotate(
        f"{performance.iloc[-1]['steady_speedup']:.1f}× speed-up",
        xy=(path_values[-1], performance.iloc[-1]["gpu_steady_seconds"]),
        xytext=(-80, 23), textcoords="offset points",
        color=REPORT_COLORS["blue"], fontweight="semibold",
        arrowprops={"arrowstyle": "->", "color": REPORT_COLORS["blue"], "lw": 1.0}
    )

    positions = np.arange(len(method_errors))
    observed_errors = np.maximum(
        method_errors["max_absolute_difference"].to_numpy(), 1e-18
    )
    axes[1].bar(
        positions, observed_errors, width=0.58,
        color=REPORT_COLORS["teal"], edgecolor="white", linewidth=0.7,
        label="Observed maximum difference"
    )
    axes[1].scatter(
        positions, method_errors["tolerance"], marker="_", s=620,
        linewidths=3.0, color=REPORT_COLORS["red"],
        label="Acceptance tolerance", zorder=3
    )
    axes[1].set_yscale("log")
    axes[1].set_xticks(
        positions, ["M0", "M1", "Batched\nGreeks", "M2", "M3"]
    )
    axes[1].set_ylabel("Absolute CPU/GPU difference (log scale)")
    axes[1].set_title("(b) Numerical-equivalence gates", loc="left")
    axes[1].legend(loc="upper left", fontsize=8.2)
    axes[1].text(
        0.98, 0.04, "All gates pass", transform=axes[1].transAxes,
        ha="right", va="bottom", color=REPORT_COLORS["teal"],
        fontsize=9, fontweight="semibold"
    )

    fig.suptitle(
        "CPU/GPU Numerical Equivalence and Scalable Acceleration",
        fontsize=14, fontweight="semibold", y=1.03
    )
    fig.text(
        0.5, -0.035,
        "Shared float64 inputs. GPU runtime includes transfer and payoff reduction; random-input generation is reported separately.",
        ha="center", fontsize=8, color=REPORT_COLORS["slate"]
    )
    fig.savefig(
        OUTPUT_DIR / "cpu_gpu_core_evidence.png",
        dpi=300, bbox_inches="tight", pad_inches=0.12
    )
    plt.show()

# Direct-GPU wall-time decomposition.
timing_plot = performance.copy()
timing_plot["other_setup_seconds"] = np.maximum(
    timing_plot["gpu_steady_seconds"]
    - timing_plot[[
        "gpu_transfer_seconds", "gpu_pricing_kernel_seconds", "gpu_reduction_seconds"
    ]].sum(axis=1),
    0.0,
)
timing_labels = [
    "Host-to-device transfer", "Pricing kernel", "Payoff reduction", "Setup / other"
]
timing_columns = [
    "gpu_transfer_seconds", "gpu_pricing_kernel_seconds",
    "gpu_reduction_seconds", "other_setup_seconds",
]
timing_colors = [
    "#80C4E9", REPORT_COLORS["blue"], "#725AC1", REPORT_COLORS["light_slate"]
]
with plt.rc_context(REPORT_STYLE):
    fig, ax = plt.subplots(figsize=(9.4, 4.55), constrained_layout=True)
    positions = np.arange(len(timing_plot))
    bottom = np.zeros(len(timing_plot))
    for column, label, color in zip(timing_columns, timing_labels, timing_colors):
        values = timing_plot[column].to_numpy()
        ax.bar(
            positions, values, bottom=bottom, width=0.62,
            label=label, color=color, edgecolor="white", linewidth=0.6
        )
        bottom += values
    for position, total in zip(positions, bottom):
        ax.text(
            position, total + max(bottom) * 0.018, f"{total:.3f}s",
            ha="center", va="bottom", fontsize=8
        )
    ax.set_xticks(positions, path_labels)
    ax.set_xlabel("Number of paths, N")
    ax.set_ylabel("GPU steady-state wall time (seconds)")
    ax.set_title("Direct GPU Runtime Decomposition", pad=10)
    ax.legend(loc="upper left", ncol=2, fontsize=8.2)
    ax.grid(axis="x", visible=False)
    fig.text(
        0.5, -0.03,
        "Transfer and reduction remain inside the reported GPU total; random-input generation is shown separately in the audit table.",
        ha="center", fontsize=8, color=REPORT_COLORS["slate"]
    )
    fig.savefig(
        OUTPUT_DIR / "gpu_timing_breakdown.png",
        dpi=300, bbox_inches="tight", pad_inches=0.12
    )
    plt.show()

# Paired CPU/GPU Greek markers: small horizontal offsets keep coincident values visible.
with plt.rc_context(REPORT_STYLE):
    fig, axes = plt.subplots(1, 3, figsize=(12.2, 4.35), constrained_layout=True)
    for axis, greek_name in zip(axes, ("Delta", "Vega", "Gamma")):
        subset = greek_agreement[
            greek_agreement["greek"] == greek_name
        ].sort_values("asset")
        positions = np.arange(len(subset))
        for position, cpu_value, gpu_value in zip(
            positions, subset["value_cpu"], subset["value_gpu"]
        ):
            axis.plot(
                [position - 0.055, position + 0.055], [cpu_value, gpu_value],
                color=REPORT_COLORS["light_slate"], linewidth=1.2, zorder=1
            )
        axis.scatter(
            positions - 0.055, subset["value_cpu"], s=48,
            facecolors="white", edgecolors=REPORT_COLORS["slate"],
            linewidths=1.5, marker="o", label="CPU", zorder=3
        )
        axis.scatter(
            positions + 0.055, subset["value_gpu"], s=42,
            color=REPORT_COLORS["blue"], marker="D", label="GPU", zorder=3
        )
        axis.set_xticks(
            positions, [f"Asset {int(value)}" for value in subset["asset"]]
        )
        axis.set_title(greek_name)
        axis.axhline(0.0, color="#94A3B8", linewidth=0.8)
        axis.grid(axis="x", visible=False)
    axes[0].set_ylabel("Finite-difference estimate")
    handles, legend_labels = axes[-1].get_legend_handles_labels()
    fig.legend(
        handles, legend_labels, loc="upper center", bbox_to_anchor=(0.5, 1.075),
        ncol=2, frameon=False
    )
    fig.suptitle(
        "Batched CRN Greeks: CPU/GPU Agreement",
        fontsize=14, fontweight="semibold", y=1.16
    )
    fig.text(
        0.5, -0.03,
        f"Maximum absolute CPU/GPU difference across all reported Greeks: {greek_agreement['absolute_difference'].max():.2e}.",
        ha="center", fontsize=8, color=REPORT_COLORS["slate"]
    )
    fig.savefig(
        OUTPUT_DIR / "batched_greeks_agreement.png",
        dpi=300, bbox_inches="tight", pad_inches=0.12
    )
    plt.show()

presentation_lines = [
    "# GPU implementation presentation guide",
    "",
    "## Implementation focus",
    "",
    "1. **Correctness before speed.** CPU and GPU consume identical float64 normal inputs and are checked by payoff component, probability mass and fair-coupon identity.",
    "2. **Break-even, not headline marketing.** Cold/shape-setup and steady-state timings are separate; H2D transfer and reduction stay inside GPU wall time.",
    "3. **Batched Greeks.** Thirteen CRN base/plus/minus scenarios share random input and correlated path work on a GPU scenario axis.",
    "4. **Honest M2/M3 boundary.** Endpoint GBM, marginal exits and nested q3 run on CUDA; exact non-integer Bessel h_ij remains in SciPy CPU because CuPy 14.1 has no ive.",
    "5. **No method substitution.** The established contract, event clocks, Brownian-bridge interface, bump convention and result schema remain unchanged.",
    "",
    "## Core chart",
    "",
    f"Use `cpu_gpu_core_evidence.png` as the primary presentation figure. Its left panel shows the measured break-even (N = {break_even_n:,}) and {performance.iloc[-1]['steady_speedup']:.2f}× steady-state speed-up at N = {int(performance.iloc[-1]['n_paths']):,}; its right panel shows that every CPU/GPU numerical-equivalence error is below its frozen tolerance.",
    "",
    "## Supporting charts",
    "",
    "- `gpu_timing_breakdown.png`: shows that transfer and reduction are included and identifies the dominant direct-pricing stage.",
    "- `batched_greeks_agreement.png`: compares CPU/GPU Delta, Vega and Gamma for all three assets.",
    "- `conditioned_pipeline_timing.png`: shows the M2/M3 hybrid boundary and retained exact CPU Bessel cost.",
    "- `cpu_gpu_performance_scaling.png`: provides the complete cold-start versus steady-state scaling view.",
    "",
    "## Headline validated numbers",
    "",
    f"- Validation gates: {int(gates['pass'].sum())}/{len(gates)} passed.",
    f"- M0/M1 maximum component difference: {direct_correctness['absolute_difference'].max():.3e}.",
    f"- M2/M3 maximum component difference: {conditioned_agreement['absolute_difference'].max():.3e}.",
    f"- Batched Greek maximum difference: {greek_agreement['absolute_difference'].max():.3e}.",
    "- Regression tests: 19 passed.",
]
presentation_guide = "\n".join(presentation_lines) + "\n"
(OUTPUT_DIR / "DAY8_PRESENTATION_GUIDE.md").write_text(
    presentation_guide, encoding="utf-8"
)
print(presentation_guide)

In [ ]:
run_manifest = pd.DataFrame([
    {"field": "status", "value": overall_status},
    {"field": "executed_at", "value": pd.Timestamp.now().isoformat()},
    {"field": "device", "value": environment_record["device_name"]},
    {"field": "precision", "value": DAY8["precision"]},
    {"field": "direct_correctness_paths", "value": correctness_cfg["n_paths"]},
    {"field": "direct_correctness_steps_per_year", "value": correctness_cfg["steps_per_year"]},
    {"field": "performance_steps_per_year", "value": performance_cfg["steps_per_year"]},
    {"field": "performance_path_grid", "value": ";".join(map(str, performance_cfg["path_grid"]))},
    {"field": "steady_state_break_even_n", "value": break_even_n},
    {"field": "maximum_steady_speedup", "value": performance["steady_speedup"].max()},
    {"field": "maximum_tested_paths", "value": performance["n_paths"].max()},
    {"field": "maximum_gpu_memory_pool_bytes", "value": performance["gpu_peak_memory_pool_bytes"].max()},
    {"field": "batched_greek_scenarios", "value": len(labels)},
    {"field": "m2_m3_probability_boundary", "value": DAY8["implementation_boundary"]["m2_m3"]},
    {"field": "speedup_gate", "value": "reported, not forced"},
    {"field": "literature_speedup_target", "value": "none"},
])
run_manifest.to_csv(OUTPUT_DIR / "run_manifest.csv", index=False)

audit_rows = []
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file() and path.name != "audit_inventory.csv":
        audit_rows.append({
            "file": path.name,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })
audit_inventory = pd.DataFrame(audit_rows)
audit_inventory.to_csv(OUTPUT_DIR / "audit_inventory.csv", index=False)
display(run_manifest)
display(audit_inventory)

## Takeaways

The executed outputs above support five conclusions:

1. CPU and GPU preserve the same M0/M1 component decomposition under shared
   input; the GPU result is not merely close in total price.
2. The measured break-even is hardware- and workload-specific. Cold-start
   GPU timing is materially worse at small N, so compile/setup must remain
   visible.
3. Batched CRN bumps reproduce CPU Delta/Vega/Gamma while amortising shared
   random input and path work.
4. M2/M3 numerical agreement passes, but the exact Bessel pair probability
   remains the documented CPU portion of a hybrid pipeline. Replacing it
   would require a separately validated GPU special-function implementation.
5. Day 8 is an engineering contribution built on the Day 5–7 mathematics;
   it does not claim that the cited papers directly solve the project's
   three-asset worst-of continuous-KI autocallable.